In [1]:
import os
import numpy as np
import scipy.io as sio
from tqdm import tqdm
import h5py

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split

import matplotlib.pyplot as plt

import sys
sys.path.append("..") 

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"   # the GPU you picked

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

print(device)

cuda


In [2]:
from processing import  load_UCI_dataset, plot_regression_results, print_metrics, plot_convtran_loss
from UCI.pre_processing import  create_dataloaders

In [3]:
BASE_DIR = "/data1/yashvi_bhuva/BP_estimation_using_PPG/VitalDB/ML/non_fiducial_features/8sec_50%overlap/vitaldb_checkpoints"
X_train, y_train, X_val, y_val, X_test, y_test = load_UCI_dataset(WINDOW_SIZE= 1000, STEP_SIZE= 500)

Total recordings: 12000
Train recordings: 9600
Validation recordings: 1200
Test recordings: 1200


100%|██████████| 9600/9600 [02:00<00:00, 79.60it/s] 


Skipped recordings: 103


100%|██████████| 1200/1200 [00:15<00:00, 76.66it/s]


Skipped recordings: 8


100%|██████████| 1200/1200 [00:15<00:00, 76.01it/s]

Skipped recordings: 9


In [4]:
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("X_val:", X_val.shape)
print("y_val:", y_val.shape)

print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

# The shape should be : (batch, channels, length), length = 8*125 = 1000samples, only one channel as PPG and 

X_train: (517037, 1, 1000)
y_train: (517037, 2)
X_val: (65284, 1, 1000)
y_val: (65284, 2)
X_test: (65554, 1, 1000)
y_test: (65554, 2)


In [5]:
print(X_train.dtype)
print(y_train.dtype)


float32
float32


In [6]:
cat ConvTran/Models/model.py

import numpy as np
from torch import nn
from Models.AbsolutePositionalEncoding import tAPE, AbsolutePositionalEncoding, LearnablePositionalEncoding
from Models.Attention import Attention, Attention_Rel_Scl, Attention_Rel_Vec


def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


class Permute(nn.Module):
    def forward(self, x):
        return x.permute(1, 0, 2)


def model_factory(config):
    if config['Net_Type'][0] == 'T':
        model = Transformer(config, output_size=config['output_size'])
    elif config['Net_Type'][0] == 'CC-T':
        model = CasualConvTran(config, output_size=config['output_size'])
    else:
        model = ConvTran(config, output_size=config['output_size'])
    return model


class Transformer(nn.Module):
    def __init__(self, config, output_size):
        super().__init__()
        # Parameters Initialization -----------------------------------------------
        channel_size, seq_len = config['Data_sh

In [7]:
train_loader, val_loader, test_loader = create_dataloaders(
    X_train,
    y_train,
    X_val,
    y_val,
    X_test,
    y_test,
    batch_size=64
)

X_batch, y_batch = next(iter(train_loader))

print("X:", X_batch.shape, X_batch.dtype)
print("y:", y_batch.shape, y_batch.dtype)

X: torch.Size([64, 1, 1000]) torch.float32
y: torch.Size([64, 2]) torch.float32


In [8]:
config = {
    # Input
    'Data_shape': (32, 1, 1000),

    # ConvTran architecture
    'emb_size': 16,
    'num_heads': 8,
    'dim_ff': 256,

    # Positional encoding
    'Fix_pos_encode': 'tAPE',
    'Rel_pos_encode': 'eRPE',

    # Dropout
    'dropout': 0.01,

    # Regression
    'output_size': 2,

    # Model type
    'Net_Type': ['C-T'],
}

In [9]:
import sys

sys.path.insert(
    0,
    "/data1/yashvi_bhuva/BP_estimation_using_PPG/ConvTran/ConvTran"
)
from Models.model import model_factory
model = model_factory(config)

X_batch, y_batch = next(iter(train_loader))

output = model(X_batch)

print("Input :", X_batch.shape)
print("Output:", output.shape)

Input : torch.Size([64, 1, 1000])
Output: torch.Size([64, 2])


In [10]:
import torch.nn as nn

criterion = nn.SmoothL1Loss()

loss = criterion(output, y_batch)

print("Prediction shape:", output.shape)
print("Target shape:", y_batch.shape)
print("Loss:", loss.item())

loss.backward()

print("Backward pass successful")

Prediction shape: torch.Size([64, 2])
Target shape: torch.Size([64, 2])
Loss: 94.56288146972656


Backward pass successful


In [11]:
from torch.optim import AdamW

optimizer = AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

optimizer.step()

print("Optimizer step successful")

Optimizer step successful


In [12]:
model.train()

X_batch, y_batch = next(iter(train_loader))

optimizer.zero_grad()

pred = model(X_batch)

loss = criterion(pred, y_batch)

loss.backward()

optimizer.step()

print("Loss:", loss.item())
print("Training step successful")

Loss: 98.81063842773438
Training step successful


In [13]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = model_factory(config).to(device)

In [14]:
loss_module = torch.nn.SmoothL1Loss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

In [15]:
from Training import SupervisedTrainer, validate, train_runner
trainer = SupervisedTrainer(
    model=model,
    dataloader=train_loader,
    device=device,
    loss_module=loss_module,
    optimizer=optimizer,
    l2_reg=None
)

val_evaluator = SupervisedTrainer(
    model=model,
    dataloader=val_loader,
    device=device,
    loss_module=loss_module,
    optimizer=None,
    l2_reg=None
)

In [16]:
import os

config['optimizer'] = optimizer
config['loss_module'] = loss_module
config['patience'] = 5
config['key_metric'] = 'loss'
config['save_dir'] = './checkpoints'

os.makedirs(config['save_dir'], exist_ok=True)

In [ ]:
config['epochs'] = 50

history = train_runner(
    config=config,
    model=model,
    trainer=trainer,
    val_evaluator=val_evaluator,
    path='./checkpoints/best_model.pth'
)

Training Epoch:   0%|          | 0/50 [00:00<?, ?it/s]2026-10-11 22:49:51,780 | INFO : Validation Summary: epoch: 1.000000 | loss: 55.605477 | SBP_MAE: 87.819725 | DBP_MAE: 24.391226 | SBP_RMSE: 90.659164 | DBP_RMSE: 26.790579 | 
2026-10-11 22:49:51,921 | INFO : Epoch 1 Training Summary: epoch: 1.000000 | loss: 78.908914 | SBP_MAE: 111.401718 | DBP_MAE: 47.416100 | SBP_RMSE: 114.160583 | DBP_RMSE: 50.092915 | 
Training Epoch:   2%|▏         | 1/50 [14:50<12:06:58, 890.18s/it]


Best validation loss: 55.60547687162861
Saving best model for epoch: 1



2026-10-11 23:04:42,264 | INFO : Validation Summary: epoch: 2.000000 | loss: 13.476860 | SBP_MAE: 19.482622 | DBP_MAE: 8.453713 | SBP_RMSE: 25.156258 | DBP_RMSE: 11.198882 | 
2026-10-11 23:04:42,401 | INFO : Epoch 2 Training Summary: epoch: 2.000000 | loss: 30.144649 | SBP_MAE: 51.153271 | DBP_MAE: 10.123207 | SBP_RMSE: 59.392471 | DBP_RMSE: 13.663148 | 
Training Epoch:   4%|▍         | 2/50 [29:40<11:52:17, 890.35s/it]


Best validation loss: 13.47686045938654
Saving best model for epoch: 2



2026-10-11 23:19:33,260 | INFO : Validation Summary: epoch: 3.000000 | loss: 11.504839 | SBP_MAE: 16.030186 | DBP_MAE: 7.956179 | SBP_RMSE: 20.577122 | DBP_RMSE: 11.167326 | 
2026-10-11 23:19:33,432 | INFO : Epoch 3 Training Summary: epoch: 3.000000 | loss: 11.724041 | SBP_MAE: 16.715088 | DBP_MAE: 7.711185 | SBP_RMSE: 21.318581 | DBP_RMSE: 10.555405 | 
Training Epoch:   6%|▌         | 3/50 [44:31<11:37:41, 890.66s/it]

...


Best validation loss: 9.293837441422836
Saving best model for epoch: 20



2026-10-12 03:46:50,045 | INFO : Validation Summary: epoch: 21.000000 | loss: 9.612410 | SBP_MAE: 13.488411 | DBP_MAE: 6.708064 | SBP_RMSE: 17.953030 | DBP_RMSE: 9.240542 | 
2026-10-12 03:46:50,049 | INFO : Epoch 21 Training Summary: epoch: 21.000000 | loss: 8.525968 | SBP_MAE: 11.942264 | DBP_MAE: 6.075874 | SBP_RMSE: 16.323755 | DBP_RMSE: 8.829718 | 
Training Epoch:  42%|████▏     | 21/50 [5:11:48<7:10:35, 890.87s/it]

2026-10-12 04:01:40,712 | INFO : Validation Summary: epoch: 22.000000 | loss: 9.597865 | SBP_MAE: 13.490439 | DBP_MAE: 6.677475 | SBP_RMSE: 17.803684 | DBP_RMSE: 9.405780 | 
2026-10-12 04:01:40,716 | INFO : Epoch 22 Training Summary: epoch: 22.000000 | loss: 8.486160 | SBP_MAE: 11.888129 | DBP_MAE: 6.050084 | SBP_RMSE: 16.262072 | DBP_RMSE: 8.804302 | 
Training Epoch:  44%|████▍     | 22/50 [5:26:38<6:55:42, 890.81s/it]

2026-10-12 04:16:31,395 | INFO : Validation Summary: epoch: 23.000000 | loss: 9.171923 | SBP_MAE: 12.850487 | DBP_MAE: 6.463447 | SBP_RMSE: 17.399862 | DBP_RMSE: 9.193261 | 
2026-10-12 04:16:31,473 | INFO : Epoch 23 Training Summary: epoch: 23.000000 | loss: 8.448331 | SBP_MAE: 11.833764 | DBP_MAE: 6.028562 | SBP_RMSE: 16.200745 | DBP_RMSE: 8.777266 | 
Training Epoch:  46%|████▌     | 23/50 [5:41:29<6:40:51, 890.79s/it]


Best validation loss: 9.171923150882426
Saving best model for epoch: 23



In [ ]:
plot_convtran_loss(history)

In [ ]:
checkpoint = torch.load(
    './checkpoints/model_best.pth',
    map_location=device
)

model.load_state_dict(checkpoint['state_dict'])
model.to(device)

test_evaluator = SupervisedTrainer(
    model=model,
    dataloader=test_loader,
    device=device,
    loss_module=loss_module,
    optimizer=None,
    l2_reg=None,
    print_interval=10,
    console=True,
    print_conf_mat=False
)

In [ ]:
test_metrics, test_results = test_evaluator.evaluate(
    epoch_num=None,
    keep_all=True
)

In [ ]:
y_true = test_results['targets']
y_pred = test_results['predictions']

print("True shape:", y_true.shape)
print("Pred shape:", y_pred.shape)

In [ ]:
# Convert to NumPy arrays if they are PyTorch tensors
if torch.is_tensor(y_true):
    y_true = y_true.detach().cpu().numpy()

if torch.is_tensor(y_pred):
    y_pred = y_pred.detach().cpu().numpy()

# Extract SBP and DBP separately
sbp_true = y_true[:, 0]
sbp_pred = y_pred[:, 0]

dbp_true = y_true[:, 1]
dbp_pred = y_pred[:, 1]

# SBP metrics and plots
print("===== SBP Metrics =====")
print_metrics(sbp_true, sbp_pred)

plot_regression_results(
    sbp_true,
    sbp_pred,
    model_name="ConvTran",
    target_name="SBP"
)

# DBP metrics and plots
print("===== DBP Metrics =====")
print_metrics(dbp_true, dbp_pred)

plot_regression_results(
    dbp_true,
    dbp_pred,
    model_name="ConvTran",
    target_name="DBP"
)

In [ ]:
from processing import evaluate_convtran_vitaldb
convtran_vitaldb_results = evaluate_convtran_vitaldb(
    model=model,  # Your trained, best-checkpoint ConvTran model
    base_dir=BASE_DIR,
    batch_size=32,
    device=device
)

Number of VitalDB cases: 1962


ConvTran VitalDB prediction:   0%|          | 0/1962 [00:00<?, ?case/s][W1011 22:22:33.912729945 CUDACachingAllocator.cpp:3933] memory allocation failed with OOM on device 0 while trying to allocate 1025507328 bytes (free: 627310592, total: 50899386368).
[W1011 22:22:33.912926775 CUDACachingAllocator.cpp:3933] memory allocation failed with OOM on device 0 while trying to allocate 1025507328 bytes (free: 627310592, total: 50899386368).
ConvTran VitalDB prediction:   0%|          | 0/1962 [00:00<?, ?case/s]


RuntimeError: NVML_SUCCESS == DriverAPI::get()->nvmlInit_v2_() INTERNAL ASSERT FAILED at "/__w/pytorch/pytorch/c10/cuda/CUDACachingAllocator.cpp":1377, please report a bug to PyTorch. 